# Support Vector Regression (SVR) with RBF Kernel

This notebook implements Support Vector Regression using the Radial Basis Function (RBF) kernel for predicting Kbearing_c values. SVR is particularly effective for non-linear regression problems by mapping data to high-dimensional spaces.

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
import time
import json

# Define Mean Absolute Percentage Error function
def mean_absolute_percentage_error(y_true, y_pred):
    """
    Calculate Mean Absolute Percentage Error (MAPE)
    MAPE = 100 * mean(|y_true - y_pred| / |y_true|)
    """
    return 100 * np.mean(np.abs((y_true - y_pred) / y_true))

In [4]:
# Load the same dataset as evaluate_models.ipynb
dataset_path = "../data/ml_formatted_datasets/standard_hokajGenTest_four_params_Kbearing_c_outliersRemoved_finiteWidthFiltered15"
scaler_directory_path = "../models/v16_activations_outliersRemoved_finiteWidthFiltered15/"

# Load train/test splits
X_train = np.load(f"{dataset_path}/X_train.npy")
y_train = np.load(f"{dataset_path}/y_train.npy")
X_test = np.load(f"{dataset_path}/X_test.npy")
y_test = np.load(f"{dataset_path}/y_test.npy")

# Flatten y arrays if needed
y_train = y_train.ravel()
y_test = y_test.ravel()

# Define feature names (only 4 features: w/r, a/c, a/t, R/t)
feature_names = ['w_over_r', 'a_over_c', 'a_over_t', 'R_over_t']

# Load scaler parameters from JSON (same ones used during training)
with open(f"{scaler_directory_path}x_scaler_params.json", 'r') as f:
    x_scaler_params = json.load(f)
with open(f"{scaler_directory_path}y_scaler_params.json", 'r') as f:
    y_scaler_params = json.load(f)

# Reconstruct scalers from saved parameters
scaler_X = StandardScaler()
scaler_X.mean_ = np.array(x_scaler_params['mean'])
scaler_X.std = np.array(x_scaler_params['std'])

scaler_y = StandardScaler()
scaler_y.mean_ = np.array(y_scaler_params['mean'])
scaler_y.std = np.array(y_scaler_params['std'])

# Apply normalization using the loaded scaler parameters
X_train_scaled = (X_train - scaler_X.mean_) / scaler_X.std
X_test_scaled = (X_test - scaler_X.mean_) / scaler_X.std
y_train_scaled = (y_train - scaler_y.mean_) / scaler_y.std
y_test_scaled = (y_test - scaler_y.mean_) / scaler_y.std

print(f"Dataset path: {dataset_path}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nFeature names: {feature_names}")
print(f"Target range: [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nScalers loaded from: {scaler_directory_path}")

Dataset path: ../data/ml_formatted_datasets/standard_hokajGenTest_four_params_Kbearing_c_outliersRemoved_finiteWidthFiltered15
X_train shape: (62478, 4)
y_train shape: (62478,)
X_test shape: (111, 4)
y_test shape: (111,)

Feature names: ['w_over_r', 'a_over_c', 'a_over_t', 'R_over_t']
Target range: [0.79, 116.90]
Training set: 62478 samples
Test set: 111 samples

Scalers loaded from: ../models/v16_activations_outliersRemoved_finiteWidthFiltered15/


## Feature Scaling

SVR is sensitive to feature scaling, so we'll standardize all features to have mean=0 and std=1.

In [5]:
# Data is already scaled using the loaded scaler parameters
# No additional scaling needed - we use the same scalers as training pipeline

print("Feature scaling completed:")
print(f"X_train scaled mean: {X_train_scaled.mean(axis=0)}")
print(f"X_train scaled std: {X_train_scaled.std(axis=0)}")
print(f"y_train scaled mean: {y_train_scaled.mean():.6f}")
print(f"y_train scaled std: {y_train_scaled.std():.6f}")

Feature scaling completed:
X_train scaled mean: [-0.20234726 -0.00905401 -0.01868836  0.06200446]
X_train scaled std: [0.03567885 0.99075036 0.94985787 1.01495984]
y_train scaled mean: -0.166447
y_train scaled std: 0.064726


In [10]:
# Subsample the scaled training data for faster hyperparameter tuning
subsample_size = 50000
X_train_subsample = X_train_scaled[:subsample_size, :]
y_train_subsample = y_train_scaled[:subsample_size]

print(f"Using {subsample_size} training samples for hyperparameter tuning")
print(f"Subsample shape: {X_train_subsample.shape}")

Using 50000 training samples for hyperparameter tuning
Subsample shape: (50000, 4)


## SVR with RBF Kernel - Baseline Model

Start with a basic SVR model using default parameters to establish baseline performance.

In [11]:
# 1. Baseline SVR with default RBF kernel
print("=== 1. Baseline SVR with RBF Kernel ===")

# Create and train baseline SVR model
svr_baseline = SVR(kernel='rbf')
start_time = time.time()
svr_baseline.fit(X_train_subsample, y_train_subsample)
training_time = time.time() - start_time

print(f"Training time: {training_time:.2f} seconds")

# Make predictions on scaled test data
y_pred_scaled_baseline = svr_baseline.predict(X_test_scaled)

# Transform predictions back to original scale using loaded scaler parameters
y_pred_baseline = y_pred_scaled_baseline * scaler_y.std + scaler_y.mean_

# Calculate metrics
r2_baseline = r2_score(y_test, y_pred_baseline)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
mape_baseline = mean_absolute_percentage_error(y_test, y_pred_baseline)

print(f"R² Score: {r2_baseline:.4f}")
print(f"MSE: {mse_baseline:.4f}")
print(f"RMSE: {np.sqrt(mse_baseline):.4f}")
print(f"MAE: {mae_baseline:.4f}")
print(f"MAPE: {mape_baseline:.2f}%")

# Display SVR parameters
print(f"\nSVR Parameters:")
print(f"  Kernel: {svr_baseline.kernel}")
print(f"  C (regularization): {svr_baseline.C}")
print(f"  Gamma: {svr_baseline.gamma}")
print(f"  Epsilon: {svr_baseline.epsilon}")
print(f"  Number of support vectors: {len(svr_baseline.support_)}")
print(f"  Support vector ratio: {len(svr_baseline.support_)/len(X_train)*100:.1f}%")

=== 1. Baseline SVR with RBF Kernel ===
Training time: 3.15 seconds
R² Score: -0.9930
MSE: 18.8150
RMSE: 4.3376
MAE: 4.0517
MAPE: 99.72%

SVR Parameters:
  Kernel: rbf
  C (regularization): 1.0
  Gamma: scale
  Epsilon: 0.1
  Number of support vectors: 156
  Support vector ratio: 0.2%
Training time: 3.15 seconds
R² Score: -0.9930
MSE: 18.8150
RMSE: 4.3376
MAE: 4.0517
MAPE: 99.72%

SVR Parameters:
  Kernel: rbf
  C (regularization): 1.0
  Gamma: scale
  Epsilon: 0.1
  Number of support vectors: 156
  Support vector ratio: 0.2%


## Hyperparameter Optimization

Use GridSearchCV to find optimal SVR parameters for better performance.

In [ ]:
# 2. Hyperparameter optimization with GridSearchCV
print("\n=== 2. SVR Hyperparameter Optimization ===")

# Define parameter grid for grid search
param_grid = {
    'C': [0.1, 1, 10, 100, 1000],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'epsilon': [0.01, 0.1, 0.2, 0.5]
}

print(f"Parameter grid size: {len(param_grid['C']) * len(param_grid['gamma']) * len(param_grid['epsilon'])} combinations")
print("Performing grid search (this may take a few minutes)...")

# Perform grid search with cross-validation
svr_grid = SVR(kernel='rbf')
grid_search = GridSearchCV(
    svr_grid, 
    param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error',
    n_jobs=-1,  # Use all available cores
    verbose=1
)

start_time = time.time()
grid_search.fit(X_train_subsample, y_train_subsample)
grid_search_time = time.time() - start_time

print(f"Grid search completed in {grid_search_time:.1f} seconds")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score (neg MSE): {grid_search.best_score_:.6f}")

# Get the best model
svr_optimized = grid_search.best_estimator_

# Make predictions with optimized model
y_pred_scaled_opt = svr_optimized.predict(X_test_scaled)
y_pred_optimized = y_pred_scaled_opt * scaler_y.scale_ + scaler_y.mean_

# Calculate metrics for optimized model
r2_optimized = r2_score(y_test, y_pred_optimized)
mse_optimized = mean_squared_error(y_test, y_pred_optimized)
mae_optimized = mean_absolute_error(y_test, y_pred_optimized)
mape_optimized = mean_absolute_percentage_error(y_test, y_pred_optimized)

print(f"\nOptimized SVR Performance:")
print(f"R² Score: {r2_optimized:.4f}")
print(f"MSE: {mse_optimized:.4f}")
print(f"RMSE: {np.sqrt(mse_optimized):.4f}")
print(f"MAE: {mae_optimized:.4f}")
print(f"MAPE: {mape_optimized:.2f}%")

print(f"\nOptimized SVR Parameters:")
print(f"  C (regularization): {svr_optimized.C}")
print(f"  Gamma: {svr_optimized.gamma}")
print(f"  Epsilon: {svr_optimized.epsilon}")
print(f"  Number of support vectors: {len(svr_optimized.support_)}")
print(f"  Support vector ratio: {len(svr_optimized.support_)/len(X_train)*100:.1f}%")

# Show improvement
print(f"\nImprovement over baseline:")
print(f"  ΔR²: {r2_optimized - r2_baseline:+.4f}")
print(f"  ΔRMSE: {np.sqrt(mse_optimized) - np.sqrt(mse_baseline):+.4f}")
print(f"  ΔMAPE: {mape_optimized - mape_baseline:+.2f}%")


=== 2. SVR Hyperparameter Optimization ===
Parameter grid size: 120 combinations
Performing grid search (this may take a few minutes)...
Fitting 5 folds for each of 120 candidates, totalling 600 fits


## Model Comparison and Visualization

In [ ]:
# 3. Model Comparison and Visualization
print("\n=== 3. Model Comparison Summary ===")

models = {
    'SVR Baseline': (r2_baseline, mse_baseline, mae_baseline, mape_baseline, y_pred_baseline),
    'SVR Optimized': (r2_optimized, mse_optimized, mae_optimized, mape_optimized, y_pred_optimized)
}

# Create comparison table
comparison_df = pd.DataFrame({
    'Model': list(models.keys()),
    'R² Score': [models[name][0] for name in models.keys()],
    'MSE': [models[name][1] for name in models.keys()],
    'RMSE': [np.sqrt(models[name][1]) for name in models.keys()],
    'MAE': [models[name][2] for name in models.keys()],
    'MAPE (%)': [models[name][3] for name in models.keys()]
})

comparison_df = comparison_df.sort_values('R² Score', ascending=False)
print(comparison_df.to_string(index=False, float_format='%.4f'))

# Visualization: Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for idx, (name, (r2, mse, mae, mape, y_pred)) in enumerate(models.items()):
    ax = axes[idx]
    
    # Predicted vs Actual scatter plot
    scatter = ax.scatter(y_test, y_pred, alpha=0.6, s=25, c='blue', edgecolors='black', linewidth=0.5)
    
    # Perfect prediction line (y=x)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=3, label='Perfect Prediction', alpha=0.8)
    
    # Calculate and plot trend line
    z = np.polyfit(y_test, y_pred, 1)
    p = np.poly1d(z)
    ax.plot(y_test, p(y_test), 'g-', alpha=0.7, linewidth=2, label=f'Trend Line (slope={z[0]:.3f})')
    
    # Formatting
    ax.set_xlabel('Actual Kbearing_c', fontsize=12, fontweight='bold')
    ax.set_ylabel('Predicted Kbearing_c', fontsize=12, fontweight='bold')
    ax.set_title(f'{name}\nR² = {r2:.4f}, RMSE = {np.sqrt(mse):.3f}, MAPE = {mape:.2f}%', 
                fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Add statistics text box
    stats_text = f'n = {len(y_test):,}\nMAE = {mae:.3f}\nMSE = {mse:.3f}'
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    # Set equal aspect ratio for better comparison
    ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.suptitle('SVR with RBF Kernel: Predicted vs Actual Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Residual analysis
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for idx, (name, (r2, mse, mae, mape, y_pred)) in enumerate(models.items()):
    ax = axes[idx]
    
    # Calculate residuals
    residuals = y_test - y_pred
    
    # Residual vs predicted plot
    ax.scatter(y_pred, residuals, alpha=0.6, s=25, c='purple', edgecolors='black', linewidth=0.5)
    
    # Add horizontal line at y=0
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2, alpha=0.8, label='Perfect Residuals')
    
    # Calculate and show residual statistics
    residual_std = np.std(residuals)
    ax.axhline(y=2*residual_std, color='orange', linestyle=':', alpha=0.7, label=f'±2σ = ±{2*residual_std:.2f}')
    ax.axhline(y=-2*residual_std, color='orange', linestyle=':', alpha=0.7)
    
    # Formatting
    ax.set_xlabel('Predicted Kbearing_c', fontsize=12, fontweight='bold')
    ax.set_ylabel('Residuals (Actual - Predicted)', fontsize=12, fontweight='bold')
    ax.set_title(f'{name} - Residual Analysis\nStd Dev = {residual_std:.3f}', 
                fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Add residual statistics
    residual_stats = f'Mean = {np.mean(residuals):.4f}\nStd = {residual_std:.4f}\nMin = {np.min(residuals):.3f}\nMax = {np.max(residuals):.3f}'
    ax.text(0.05, 0.95, residual_stats, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

plt.tight_layout()
plt.suptitle('SVR Residual Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# 4. Feature Importance Analysis for SVR
print("\n=== 4. SVR Model Analysis ===")

# Since SVR doesn't provide direct feature importance, we can analyze support vectors
print(f"Support Vector Analysis for Optimized SVR:")
print(f"  Total training samples: {len(X_train):,}")
print(f"  Number of support vectors: {len(svr_optimized.support_):,}")
print(f"  Support vector percentage: {len(svr_optimized.support_)/len(X_train)*100:.1f}%")

# Analyze support vector distribution across feature space
support_vectors = X_train_scaled[svr_optimized.support_]
print(f"\nSupport Vector Statistics (scaled features):")
for i, feature_name in enumerate(feature_names):
    sv_mean = support_vectors[:, i].mean()
    sv_std = support_vectors[:, i].std()
    total_mean = X_train_scaled[:, i].mean()
    total_std = X_train_scaled[:, i].std()
    print(f"  {feature_name:15s}: SV mean={sv_mean:+.3f} (±{sv_std:.3f}), Total mean={total_mean:+.3f} (±{total_std:.3f})")

# Performance comparison with metrics from classical regression (if available)
print(f"\n=== SVR Performance Summary ===")
print(f"Best SVR Model Performance:")
print(f"  R² Score: {r2_optimized:.4f}")
print(f"  RMSE: {np.sqrt(mse_optimized):.4f}")
print(f"  MAE: {mae_optimized:.4f}")
print(f"  MAPE: {mape_optimized:.2f}%")

print(f"\nModel Characteristics:")
print(f"  Kernel: RBF (Radial Basis Function)")
print(f"  Optimal C: {svr_optimized.C}")
print(f"  Optimal Gamma: {svr_optimized.gamma}")
print(f"  Optimal Epsilon: {svr_optimized.epsilon}")

# Final model recommendation
print(f"\n=== Model Recommendation ===")
print(f"The optimized SVR with RBF kernel achieved:")
print(f"  • R² Score of {r2_optimized:.4f} (explaining {r2_optimized*100:.1f}% of variance)")
print(f"  • MAPE of {mape_optimized:.2f}% (average error of {mape_optimized:.2f}%)")
print(f"  • Uses {len(svr_optimized.support_)/len(X_train)*100:.1f}% of training data as support vectors")
print(f"  • Training time: {grid_search_time:.1f} seconds for hyperparameter optimization")